# Gather results from AMG prediction tools and compare

In [53]:
! pip install polars pandas --quiet

In [54]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks")
GENOME_DIR = ROOT_DIR.joinpath("benchmark_genomes")
MAIN_DIR = ROOT_DIR.joinpath("metabolism_benchmark")

In [55]:
import os
AMG_TABLES_DIR = MAIN_DIR.joinpath("amg_tables")
os.makedirs(AMG_TABLES_DIR, exist_ok=True)

## Collect AMG prediction results from each tool

In [56]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [57]:
gene_id_mapping = pl.read_parquet(MAIN_DIR.joinpath("dramv_genes_reformatted/gene_id_mapping.parquet"))
# gene_id_mapping = reformat_source_ecosystem_tool(gene_id_mapping)
gene_id_mapping

subset,sample,old_gene,new_gene,old_scaffold,new_scaffold,gene_number,start,end,frame,scaffold_n_genes,scaffold_len_bases,metadata
str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,str
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_1""","""391036.SAMN02367296.CP007474_1""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",1,1,1005,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_1;conf=100.00;cscore=98.58;Dbxref=""ko:K01599"";gc_cont=0.338;pa…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_2""","""391036.SAMN02367296.CP007474_2""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",2,1102,1926,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_2;conf=100.00;cscore=48.58;Dbxref=""ko:K02276"";gc_cont=0.348;pa…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_3""","""391036.SAMN02367296.CP007474_3""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",3,1928,2398,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_3;conf=100.00;cscore=45.39;Dbxref=""vogdb:VOG40486"";gc_cont=0.3…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_4""","""391036.SAMN02367296.CP007474_4""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",4,2395,2985,-1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_4;conf=99.92;cscore=30.30;Dbxref=""ko:K08591"";gc_cont=0.286;par…"
"""complete_host_genomes""","""host_genomes_gut""","""391036.SAMN02367296.CP007474-cat_2_5""","""391036.SAMN02367296.CP007474_5""","""391036.SAMN02367296.CP007474-cat_2""","""391036.SAMN02367296.CP007474""",5,3081,3920,1,910,1148482,"""ID=391036.SAMN02367296.CP007474-cat_2_5;conf=100.00;cscore=112.25;Dbxref=""ko:K00767"";gc_cont=0.298;p…"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""viromes""","""soil_T42_15_3_45""","""scaffold_7091_c1__lt2gene-cat_1_1""","""scaffold_7091_c1__lt2gene_1""","""scaffold_7091_c1__lt2gene-cat_1""","""scaffold_7091_c1__lt2gene""",1,1,2931,-1,1,2931,"""ID=scaffold_7091_c1__lt2gene-cat_1_1;conf=99.99;cscore=347.23;Dbxref=""viral:YP_010091592.1"";gc_cont=…"
"""viromes""","""soil_T42_15_3_45""","""scaffold_7858_c1__lt2gene-cat_1_1""","""scaffold_7858_c1__lt2gene_1""","""scaffold_7858_c1__lt2gene-cat_1""","""scaffold_7858_c1__lt2gene""",1,1,1146,-1,2,2798,"""ID=scaffold_7858_c1__lt2gene-cat_1_1;conf=100.00;cscore=160.20;gc_cont=0.695;partial=10;rbs_motif=GG…"
"""viromes""","""soil_T42_15_3_45""","""scaffold_7858_c1__lt2gene-cat_1_2""","""scaffold_7858_c1__lt2gene_2""","""scaffold_7858_c1__lt2gene-cat_1""","""scaffold_7858_c1__lt2gene""",2,1716,2798,1,2,2798,"""ID=scaffold_7858_c1__lt2gene-cat_1_2;conf=100.00;cscore=173.16;gc_cont=0.635;partial=01;rbs_motif=No…"


**IMPORTANT:** When merging, make sure to merge by gene/scaffold AND sample, since some datasets use the same scaffold IDs across samples

In [58]:
def reformat_for_merge(df: pl.DataFrame) -> pl.DataFrame:
    return df.drop(["subset", "old_gene", "old_scaffold", "metadata"]).rename({
        "new_gene": "gene",
        "new_scaffold": "scaffold",
    })

### VIBRANT AMGs

In [59]:
VIBRANT_DIR = MAIN_DIR.joinpath("vibrant_outputs")

In [60]:
pattern_vibrant_individuals = "*/*/*/*_results_*/*_AMG_individuals*"
paths_vibrant = sorted(VIBRANT_DIR.glob(pattern_vibrant_individuals))

fragment_pattern = r"_fragment_\d+"

dfs_vibrant = []
for p in paths_vibrant:
    try:
        parts = p.parts
        i = parts.index("vibrant_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
        # print(f"Processing VIBRANT AMG file: {p} (source={source}, sample={sample}, ecosystem={ecosystem})")

        try:
            df = pl.read_csv(p, separator="\t", infer_schema_length=100, has_header=True)
        except Exception:
            df = pl.read_csv(p, separator=",", infer_schema_length=100, has_header=True)

        df = df.with_columns([
            pl.lit(sample).alias("sample"),
            pl.lit(source).alias("source"),
            pl.lit(ecosystem).alias("ecosystem"),
        ])

        # Identify protein ID column
        candidates = ("protein", "gene", "protein_id", "ORF", "orf", "gene_callers_id")
        prot_col = next((c for c in candidates if c in df.columns), df.columns[0])

        # Ensure prot_id is standardized
        df = df.with_columns([
            pl.col(prot_col).cast(pl.Utf8)
            .str.split(" ").list.first()
            .alias("prot_id")
        ])

        # Keep a copy of original IDs WITH fragment suffix (for later quality join)
        df = df.with_columns([
            pl.col("prot_id").alias("prot_id_with_fragment"),
            pl.when(pl.col("scaffold").is_not_null())
              .then(pl.col("scaffold").alias("scaffold_with_fragment"))
              .otherwise(pl.lit(None))
              .alias("scaffold_with_fragment")
        ])

        # Clean versions of IDs for gene info join
        df = df.with_columns([
            pl.col("prot_id").str.replace_all(fragment_pattern, "").alias("prot_id"),
            pl.when(pl.col("scaffold").is_not_null())
              .then(pl.col("scaffold").cast(pl.Utf8).str.replace_all(fragment_pattern, ""))
              .otherwise(pl.col("scaffold"))
              .alias("scaffold")
        ])

        # Join with gene info using CLEANED prot_id
        df = df.join(
            reformat_for_merge(gene_id_mapping),
            left_on=["prot_id", "sample"],
            right_on=["gene", "sample"],
            how="left"
        )

        dfs_vibrant.append(df)

    except Exception as e:
        print(f"Skipping {p}: {e}")

In [61]:
# Combine all AMG individual tables
vibrant_amgs_all = pl.concat(dfs_vibrant, how="diagonal_relaxed")

In [62]:
pattern_vibrant_quality = "*/*/*/*_results_*/*_genome_quality*"
paths_vibrant_quality = sorted(VIBRANT_DIR.glob(pattern_vibrant_quality))

dfs_vibrant_quality = []
for p in paths_vibrant_quality:
    try:
        parts = p.parts
        i = parts.index("vibrant_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem

        try:
            dfq = pl.read_csv(p, separator="\t", infer_schema_length=100, has_header=True)
        except Exception:
            dfq = pl.read_csv(p, separator=",", infer_schema_length=100, has_header=True)

        dfq = dfq.with_columns([
            pl.lit(sample).alias("sample"),
            pl.lit(source).alias("source"),
            pl.lit(ecosystem).alias("ecosystem"),
        ])
        dfs_vibrant_quality.append(dfq)
    except Exception as e:
        print(f"Skipping quality {p}: {e}")

vibrant_quality_all = pl.concat(dfs_vibrant_quality, how="diagonal_relaxed")

In [63]:
vibrant_results_raw = (
    vibrant_amgs_all.join(
        vibrant_quality_all,
        left_on=["sample", "source", "ecosystem", "scaffold_with_fragment"],
        right_on=["sample", "source", "ecosystem", "scaffold"],
        how="left"
    )
    .drop(["protein", "scaffold_right", "scaffold_with_fragment", "prot_id_with_fragment"])
    .unique()
    .rename({"prot_id": "gene"})
    .sort(["source", "sample", "ecosystem", "scaffold", "start"])
)

vibrant_results_raw = vibrant_results_raw.select(
            [
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            ] + [col for col in vibrant_results_raw.columns if col not in {
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            }]
    )

In [64]:
vibrant_results_raw

source,sample,ecosystem,gene,gene_number,start,end,frame,scaffold,scaffold_n_genes,scaffold_len_bases,AMG KO,AMG KO name,Pfam,Pfam name,type,Quality
str,str,str,str,i64,i64,i64,i64,str,i64,i64,str,str,str,str,str,str
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258_2""",2,423,1340,-1,"""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258""",54,40118,"""K01772""","""hemH, FECH; protoporphyrin/coproporphyrin ferrochelatase [EC:4.99.1.1 4.99.1.9]""","""PF00762.19""","""Ferrochelatase""","""lysogenic""","""high quality draft"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2693429676_000016_2693429676_2693458520_27402-60603_10""",10,5883,6905,1,"""IMGVR_UViG_2693429676_000016_2693429676_2693458520_27402-60603""",43,33202,"""K00558""","""DNMT1, dcm; DNA (cytosine-5)-methyltransferase 1 [EC:2.1.1.37]""","""PF00145.17""","""C-5 cytosine-specific DNA methylase""","""lysogenic""","""high quality draft"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2718218227_000005_2718218227_2718228826_2038485-2104644_74""",74,46581,47318,1,"""IMGVR_UViG_2718218227_000005_2718218227_2718228826_2038485-2104644""",94,66160,"""K21140""","""mec; [CysO sulfur-carrier protein]-S-L-cysteine hydrolase [EC:3.13.1.6]""","""PF00877.19""","""NlpC/P60 family""","""lysogenic""","""high quality draft"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2839645069_000001_2839645069_2839645076_48518-102011_47""",47,34748,35548,-1,"""IMGVR_UViG_2839645069_000001_2839645069_2839645076_48518-102011""",81,53494,"""K00390""","""cysH; phosphoadenosine phosphosulfate reductase [EC:1.8.4.8 1.8.4.10]""","""PF01507.19""","""Phosphoadenosine phosphosulfate reductase family""","""lytic""","""high quality draft"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2904879368_000003_2904879368_2904879379_176713-228057_32""",32,23908,24711,1,"""IMGVR_UViG_2904879368_000003_2904879368_2904879379_176713-228057""",73,51345,"""K00558""","""DNMT1, dcm; DNA (cytosine-5)-methyltransferase 1 [EC:2.1.1.37]""","""PF00145.17""","""C-5 cytosine-specific DNA methylase""","""lytic""","""medium quality draft"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""viromes""","""soil_T42_15_2_44""","""soil""","""scaffold_9493_c1_8""",8,4284,4598,-1,"""scaffold_9493_c1""",8,4598,"""K01923""","""purC; phosphoribosylaminoimidazole-succinocarboxamide synthase [EC:6.3.2.6]""","""PF01259.18""","""SAICAR synthetase""","""lysogenic""","""low quality draft"""
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_22_c1_40""",40,28405,29127,1,"""scaffold_22_c1""",63,47603,"""K21140""","""mec; [CysO sulfur-carrier protein]-S-L-cysteine hydrolase [EC:3.13.1.6]""","""PF00877.19""","""NlpC/P60 family""","""lysogenic""","""medium quality draft"""
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_473_c1_10""",10,3771,4970,-1,"""scaffold_473_c1""",16,11671,"""K00558""","""DNMT1, dcm; DNA (cytosine-5)-methyltransferase 1 [EC:2.1.1.37]""","""PF00145.17""","""C-5 cytosine-specific DNA methylase""","""lytic""","""low quality draft"""


In [65]:
vibrant_results_raw.write_parquet(AMG_TABLES_DIR.joinpath("vibrant_results_raw.parquet"))

### DRAM-V AMGs

In [66]:
DRAMV_DIR = MAIN_DIR.joinpath("dramv_outputs")

In [67]:
paths_dramv = sorted(DRAMV_DIR.glob("*/*/distilled/amg_summary.tsv"))

dfs = []
for p in paths_dramv:
    # print(f"Processing DRAMv AMG file: {p}")
    parts = p.parts
    try:
        i = parts.index("dramv_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
    except (ValueError, IndexError):
        continue

    df = pl.read_csv(p, separator="\t", infer_schema_length=200).with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
        pl.col("gene").str.extract(r"_(\d+)$", 1).cast(pl.Int64).alias("gene_position")
    ])
    dfs.append(df)

In [68]:
dramv_results_raw = pl.concat(dfs, how="diagonal_relaxed").unique()
dramv_results_raw = (
    dramv_results_raw.join(
        gene_id_mapping.drop("metadata", "subset", "old_scaffold"),
        left_on=["gene", "sample"],
        right_on=["old_gene", "sample"],
        how="left"
    )
    .drop(["gene", "scaffold", "gene_position"])
    .rename({"new_gene": "gene", "new_scaffold": "scaffold"})
)
dramv_results_raw = (
    dramv_results_raw.select(
            [
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            ] + [col for col in dramv_results_raw.columns if col not in {
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            }]
    )
    .sort(
        ["source", "sample", "ecosystem", "scaffold", "start"],
        descending=[False, False, False, False, False]
        )
)

In [69]:
dramv_results_raw

source,sample,ecosystem,gene,gene_number,start,end,frame,scaffold,scaffold_n_genes,scaffold_len_bases,gene_id,auxiliary_score,amg_flags,gene_description,module,sheet,header,subheader,potential_amg,gene_id_origin,metabolism,reference,verified
str,str,str,str,i64,i64,i64,i64,str,i64,i64,str,i64,str,str,str,str,str,str,bool,str,str,str,str
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_2""",2,963,1640,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""PF04896""",4,"""MKETFB""","""Ammonia monooxygenase/methane monooxygenase, subunit C""",null,null,null,null,null,"""amg_database""","""M""","""Roux et al. 2016""","""False"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_2""",2,963,1640,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""PF00317""",4,"""MKETFB""","""nrdA/nrdB; ribonucleotide reductase (RNR)""","""M00053""",null,null,null,null,"""amg_database""",null,"""Thompson et al. 2011; Sullivan et al. 2005""","""True"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_5""",5,4349,5038,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""PF00262""",4,"""MKTFB""","""Calreticulin family""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_5""",5,4349,5038,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""PF02347""",4,"""MKTFB""","""Glycine cleavage system P-protein""",null,null,null,null,null,"""amg_database""","""M""","""Roux et al. 2016""","""False"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_6""",6,5063,5374,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""PF04724""",4,"""MKETB""","""Glycosyltransferase family 17""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_9_c1_33""",33,63895,64125,-1,"""scaffold_9_c1""",34,64496,"""PF02548""",4,"""MKETFB""","""Ketopantoate hydroxymethyltransferase""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False"""
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_9_c1_33""",33,63895,64125,-1,"""scaffold_9_c1""",34,64496,"""PF02397""",4,"""MKETFB""","""Bacterial sugar transferase""",null,null,null,null,null,"""amg_database""",null,"""Roux et al. 2016""","""False"""
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_9_c1_33""",33,63895,64125,-1,"""scaffold_9_c1""",34,64496,"""PF10014""",4,"""MKETFB""","""2OG-Fe dioxygenase""",null,null,null,null,null,"""amg_database""","""M""","""Roux et al. 2016""","""False"""


In [70]:
dramv_results_raw.write_parquet(AMG_TABLES_DIR.joinpath("dramv_results_raw.parquet"))

### CheckAMG annotate AMGs

In [71]:
CHECKAMG_DIR = MAIN_DIR.joinpath("checkamg_annotate_v1.1_outputs")

In [72]:
pattern_checkamg = "*/*/results/final_results.parquet"
paths_checkamg = sorted(CHECKAMG_DIR.glob(pattern_checkamg))

dfs_checkamg = []
for p in paths_checkamg:
    # print(f"Processing CheckAMG AMG file: {p}")
    parts = p.parts
    try:
        i = parts.index("checkamg_annotate_v1.1_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
    except (ValueError, IndexError):
        continue

    df = pl.read_parquet(p)

    if df is not None:
        df = df.with_columns([
            pl.lit(sample).alias("sample"),
            pl.lit(source).alias("source"),
            pl.lit(ecosystem).alias("ecosystem")
        ])
        
        df = df.join(
            reformat_for_merge(gene_id_mapping),
            left_on=["Protein", "Contig", "sample"],
            right_on=["gene", "scaffold", "sample"],
            how="left"
            ).rename({"Protein": "gene", "Contig": "scaffold"})
        df = df.drop("Genome")
        df = df.select(
            [
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            ] + [col for col in df.columns if col not in {
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            }]
        ).sort(
            ["source", "sample", "ecosystem", "scaffold", "start"],
            descending=[False, False, False, False, False]
        )
        
        dfs_checkamg.append(df)

checkamg_results_raw = pl.concat(dfs_checkamg, how="diagonal_relaxed")

In [73]:
checkamg_results_raw

source,sample,ecosystem,gene,gene_number,start,end,frame,scaffold,scaffold_n_genes,scaffold_len_bases,Protein Classification,Protein Viral Origin Confidence,Protein in Strict Viral Region,Function,KEGG KO,KEGG KO Name,FOAM ID,FOAM Annotation,Pfam Accession,Pfam Name,CAZy Family,CAZy Activities,METABOLIC db ID,METABOLIC Annotation,CAMPER ID,CAMPER Annotation,PHROG Number,PHROG Annotation,Best Scoring HMM,Best Scoring HMM Annotation,Best Scoring HMM Origin
str,str,str,str,i64,i64,i64,i64,str,i64,i64,str,str,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_1""",1,1,966,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""unclassified""","""low""",true,"""transposase""","""K07497""","""K07497; putative transposase""",null,null,null,null,null,null,null,null,null,null,"""phrog_157""","""transposase""","""phrog_157""","""transposase""","""PHROG"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_2""",2,963,1640,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""regulatory""","""low""",true,"""Helix-turn-helix domain""",null,null,null,null,"""PF13518.13""","""Helix-turn-helix domain""",null,null,null,null,null,null,null,null,"""PF13518""","""Helix-turn-helix domain""","""Pfam"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_3""",3,1796,2809,1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""unclassified""","""low""",true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_4""",4,2657,4198,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""unclassified""","""high""",true,"""integrase""","""K06400""","""spoIVCA; site-specific DNA recombinase""",null,null,null,null,null,null,null,null,null,null,"""phrog_95""","""integrase""","""phrog_95""","""integrase""","""PHROG"""
"""complete_virus_genomes""","""virus_genomes_gut""","""gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_5""",5,4349,5038,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,"""unclassified""","""high""",true,"""""",null,null,null,null,null,null,null,null,null,null,null,null,"""phrog_10934""","""""","""phrog_10934""","""""","""PHROG"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_9_c1_30""",30,61334,62350,-1,"""scaffold_9_c1""",34,64496,"""unclassified""","""low""",true,"""cbf, cbf1; 3'-5' exoribonuclease [EC:3.1.-.-]""","""K03698""","""cbf, cbf1; 3'-5' exoribonuclease [EC:3.1.-.-]""",null,null,null,null,null,null,null,null,null,null,null,null,"""K03698""","""cbf, cbf1; 3'-5' exoribonuclease [EC:3.1.-.-]""","""KEGG"""
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_9_c1_31""",31,62347,62715,-1,"""scaffold_9_c1""",34,64496,"""unclassified""","""low""",true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""viromes""","""soil_T42_15_3_45""","""soil""","""scaffold_9_c1_32""",32,62712,63902,-1,"""scaffold_9_c1""",34,64496,"""unclassified""","""low""",true,"""int; integrase""","""K14059""","""int; integrase""",null,null,"""PF00589.28""","""Phage integrase family""",null,null,null,null,null,null,"""phrog_1""","""integrase""","""K14059""","""int; integrase""","""KEGG"""


In [74]:
checkamg_results_raw.write_parquet(AMG_TABLES_DIR.joinpath("checkamg_results_raw.parquet"))

## Format AMG prediction tables
Gene-level AMG predictions only, no functional annotations, just "was this gene a predicted AMG?"

In [75]:
def reformat_source_ecosystem_tool(df):
    df = df.with_columns([
        pl.col("ecosystem")
        .str.replace_all("gut", "Human gut", literal=True)
        .str.replace_all("freshwater", "Aquatic", literal=True)
        .str.replace_all("marine", "Aquatic", literal=True)
        .str.replace_all("soil", "Soil", literal=True)
        .str.replace_all("aquatic", "Aquatic", literal=True)
        .alias("ecosystem"),

        pl.col("source")
        .str.replace_all("metagenomes", "Mixed metagenomes", literal=True)
        .str.replace_all("viromes", "Viromes", literal=True)
        .str.replace_all("complete_virus_genomes", "Viral genomes", literal=True)
        .alias("source"),
    ])

    if "tool" in df.columns:
        df = df.with_columns([
            pl.col("tool")
            .str.replace_all("checkamg", "CheckAMG", literal=True)
            .str.replace_all("vibrant", "VIBRANT", literal=True)
            .str.replace_all("dramv", "DRAM-V", literal=True)
            .str.replace_all("DRAMV", "DRAM-V", literal=True)
            .alias("tool"),
        ])

    return df

### DRAM-V AMGs
DRAM-V AMG predictions can have a range of "auxiliary scores" and additional "AMG flags" that, together, indicate the confidence that a predicted AMG is virus encoded and auxiliary. The [DRAM-V documentation](https://github.com/WrightonLabCSU/DRAM/wiki/1.-How-DRAM-Works#dram-v-in-detail) states:

>The auxiliary scores are on a scale from 1 to 5 representing the confidence that a gene is viral in origin where a score of 1 represents a gene that is confidently viral and 5 a gene that users should take caution in treating as a viral gene. A gene is given an auxiliary score of 1 if there is at least one hallmark gene on the left and right flank. Auxiliary scores of 2 are assigned when the gene has a hallmark gene on one flank and a viral-like gene on the other flank. Auxiliary scores of 3 are assigned to genes that have a viral like gene on both flanks. An auxiliary score of 4 is given to genes with a viral-like or hallmark gene on one flank and no viral-like or hallmark gene on the other flank and all genes that are part of a stretch with three or more adjacent genes with non-viral metabolic function. An auxiliary score of 5 is given to genes on contigs with no viral-like or hallmark genes and genes on the end of contigs.
>
>...
>
>The near contig end flag (F) is given when the gene is within 5000 bases of the end of a contig. The transposon flag (T) is given when the gene is on a contig that contains a transposon.
>
>...
>
>By default, a gene is considered a potential AMG if the auxiliary score is less than 4, has been assigned an M flag, has not been assigned an A, V or T flag. 

Given this range of possibilities, a set of DRAM-V prediction levels will be established:

1. `DRAMV_default`: genes with the "M" flag, auxiliary scores < 4, no A/V/T flags
2. `DRAMV_allow_T`: genes with the "M" flag, auxiliary scores < 4, no A/V flag, T allowed
3. `DRAMV_aux_4`: genes with the "M" flag, auxiliary scores <= 4, no A/V/T flags
4. `DRAMV_aux_4_allow_T`: genes with the "M" flag, auxiliary scores <= 4, no A/V flag, T allowed

In [76]:
dramv_amg_genes = (
    dramv_results_raw
    .with_columns([
        # DRAMV_default: aux_score < 4, M flag, no T/A/V flags
        pl.when(
            (pl.col("auxiliary_score") < 4) &
            (pl.col("amg_flags").str.contains("M").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("T").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("A").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("V").fill_null(False))
        ).then(pl.lit(True)).otherwise(pl.lit(False)).alias("DRAMV_default"),

        # DRAMV_allow_T: keep aux_score < 4, M flag, allow T flag but no A/V flags
        pl.when(
            (pl.col("auxiliary_score") < 4) &
            (pl.col("amg_flags").str.contains("M").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("A").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("V").fill_null(False))
        ).then(pl.lit(True)).otherwise(pl.lit(False)).alias("DRAMV_allow_T"),

        # DRAMV_aux_4: relax aux_score to <= 4, M flag, no T/A/V flags
        pl.when(
            (pl.col("auxiliary_score") <= 4) &
            (pl.col("amg_flags").str.contains("M").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("T").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("A").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("V").fill_null(False))
        ).then(pl.lit(True)).otherwise(pl.lit(False)).alias("DRAMV_aux_4"),

        # DRAMV_aux_4_allow_T: relax aux_score to <= 4, M flag, allow T flag but no A/V flags
        pl.when(
            (pl.col("auxiliary_score") <= 4) &
            (pl.col("amg_flags").str.contains("M").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("A").fill_null(False)) &
            (~pl.col("amg_flags").str.contains("V").fill_null(False))
        ).then(pl.lit(True)).otherwise(pl.lit(False)).alias("DRAMV_aux_4_allow_T"),

        # Individual flag booleans
        # Explicit boolean columns for each DRAM-V flag letter so downstream
        # code can filter/group without string parsing.
        pl.col("auxiliary_score").alias("DRAMV_auxiliary_score"),
        pl.col("amg_flags").str.contains("M").fill_null(False).alias("DRAMV_M"),
        pl.col("amg_flags").str.contains("T").fill_null(False).alias("DRAMV_T"),
        pl.col("amg_flags").str.contains("V").fill_null(False).alias("DRAMV_V"),
        pl.col("amg_flags").str.contains("A").fill_null(False).alias("DRAMV_A"),
        pl.col("amg_flags").str.contains("P").fill_null(False).alias("DRAMV_P"),
        pl.col("amg_flags").str.contains("B").fill_null(False).alias("DRAMV_B"),
        pl.col("amg_flags").str.contains("F").fill_null(False).alias("DRAMV_F"),
    ])
    .drop(["auxiliary_score", "amg_flags"])
)

In [77]:
dramv_amg_genes = (
    dramv_amg_genes
    .select(
        [
            "gene",
            "scaffold",
            "source",
            "sample",
            "ecosystem",
        ] + [col for col in dramv_amg_genes.columns if col.startswith("DRAMV_")]
    )
    .unique() # deduplicate because of multiple annotations to the same gene
    .sort(["source", "sample", "ecosystem", "gene"])
)
dramv_amg_genes = reformat_source_ecosystem_tool(dramv_amg_genes)

In [78]:
dramv_amg_genes

gene,scaffold,source,sample,ecosystem,DRAMV_default,DRAMV_allow_T,DRAMV_aux_4,DRAMV_aux_4_allow_T,DRAMV_auxiliary_score,DRAMV_M,DRAMV_T,DRAMV_V,DRAMV_A,DRAMV_P,DRAMV_B,DRAMV_F
str,str,str,str,str,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool
"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_10""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,false,true,4,true,true,false,false,false,true,false
"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_11""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,false,true,4,true,true,false,false,false,true,false
"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_12""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,false,true,4,true,true,false,false,false,true,false
"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_14""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,false,true,4,true,true,false,false,false,true,false
"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_15""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,false,true,4,true,true,false,false,false,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_5""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,false,true,4,true,true,false,false,false,true,false
"""scaffold_9_c1_6""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,false,true,4,true,true,false,false,false,true,false
"""scaffold_9_c1_7""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,false,true,4,true,true,false,false,false,true,false


In [79]:
dramv_amg_genes.write_parquet(AMG_TABLES_DIR.joinpath("amg_predictions_dramv.parquet"))

### CheckAMG annotate
CheckAMG annotate assigns "high", "medium", and "low" confidence viral predictions. These predictions are independent of AMG predictions. Genes will inherit their viral origin confidence predictions from CheckAMG, with the following labels:

1. `CheckAMG_high`: high confidence only, no medium or low
2. `CheckAMG_medium`: high and medium confidence, no low
3. `CheckAMG_low`: high, medium and low confidence predictions (all)

All genes will be required to have "metabolic" under "Protein Classification", indicating it is a predicted AMG and not unclassified.

In [80]:
checkamg_amg_genes = (
    checkamg_results_raw
    .filter(pl.col("Protein Classification") == "metabolic")

    .with_columns([        
        pl.when(pl.col("Protein Viral Origin Confidence") == "high")
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("CheckAMG_high"),

        pl.when(pl.col("Protein Viral Origin Confidence").is_in(["high", "medium"]))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("CheckAMG_medium"),

        pl.when(pl.col("Protein Viral Origin Confidence").is_in(["high", "medium", "low"]))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("CheckAMG_low"),
    ])
)

checkamg_amg_genes = (
    checkamg_amg_genes
    .select(
        [
            "gene",
            "scaffold",
            "source",
            "sample",
            "ecosystem",
        ] + [col for col in checkamg_amg_genes.columns if col.startswith("CheckAMG_")]
    )
    .unique() # deduplicate in case of any redundant joins
    .sort(["source", "sample", "ecosystem", "gene"])
)

checkamg_amg_genes = reformat_source_ecosystem_tool(checkamg_amg_genes)

In [81]:
checkamg_amg_genes

gene,scaffold,source,sample,ecosystem,CheckAMG_high,CheckAMG_medium,CheckAMG_low
str,str,str,str,str,bool,bool,bool
"""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258_2""","""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,true
"""IMGVR_UViG_2548876954_000011_2548876954_2549006873_262978-300842_47""","""IMGVR_UViG_2548876954_000011_2548876954_2549006873_262978-300842""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",true,true,true
"""IMGVR_UViG_2548877045_000002_2548877045_2549042790_110417-137006_22""","""IMGVR_UViG_2548877045_000002_2548877045_2549042790_110417-137006""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,true,true
"""IMGVR_UViG_2548877045_000002_2548877045_2549042790_110417-137006_35""","""IMGVR_UViG_2548877045_000002_2548877045_2549042790_110417-137006""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,true,true
"""IMGVR_UViG_2802429512_000001_2802429512_2802484553_443685-483098_45""","""IMGVR_UViG_2802429512_000001_2802429512_2802484553_443685-483098""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,false,true
…,…,…,…,…,…,…,…
"""scaffold_9_c1_12""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,true
"""scaffold_9_c1_13""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,true
"""scaffold_9_c1_16""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,true


In [82]:
checkamg_amg_genes.write_parquet(AMG_TABLES_DIR.joinpath("amg_predictions_checkamg.parquet"))

### VIBRANT
VIBRANT results should already only contain AMG predictions from where the tables were sourced, above, but will still be filtered so "AMG KO" is not null, just in case. The genome quality predictions will be used to label viral confidence:

1. `VIBRANT_high`: "complete circular" or "high quality draft" genome
2. `VIBRANT_medium`: "complete circular", "high quality draft", or "medium quality draft" genome
3. `VIBRANT_low`: "complete circular", "high quality draft", "medium quality draft", or "low quality draft" genome

In [83]:
vibrant_amg_genes = (
    vibrant_results_raw

    .filter(~pl.col("AMG KO").is_null())

    .with_columns([        
        pl.when(pl.col("Quality").is_in(["complete circular", "high quality draft"]))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("VIBRANT_high"),

        pl.when(pl.col("Quality").is_in(["complete circular", "high quality draft", "medium quality draft"]))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("VIBRANT_medium"),

        pl.when(pl.col("Quality").is_in(["complete circular", "high quality draft", "medium quality draft", "low quality draft"]))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("VIBRANT_low"),
    ])
)

vibrant_amg_genes = (
    vibrant_amg_genes
    .select(
        [
            "gene",
            "scaffold",
            "source",
            "sample",
            "ecosystem",
        ] + [col for col in vibrant_amg_genes.columns if col.startswith("VIBRANT_")]
    )
    .unique() # deduplicate in case of any redundant joins
    .sort(["source", "sample", "ecosystem", "gene"])
)

vibrant_amg_genes = reformat_source_ecosystem_tool(vibrant_amg_genes)

In [84]:
vibrant_amg_genes

gene,scaffold,source,sample,ecosystem,VIBRANT_high,VIBRANT_medium,VIBRANT_low
str,str,str,str,str,bool,bool,bool
"""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258_2""","""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",true,true,true
"""IMGVR_UViG_2693429676_000016_2693429676_2693458520_27402-60603_10""","""IMGVR_UViG_2693429676_000016_2693429676_2693458520_27402-60603""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",true,true,true
"""IMGVR_UViG_2718218227_000005_2718218227_2718228826_2038485-2104644_74""","""IMGVR_UViG_2718218227_000005_2718218227_2718228826_2038485-2104644""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",true,true,true
"""IMGVR_UViG_2839645069_000001_2839645069_2839645076_48518-102011_47""","""IMGVR_UViG_2839645069_000001_2839645069_2839645076_48518-102011""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",true,true,true
"""IMGVR_UViG_2904879368_000003_2904879368_2904879379_176713-228057_32""","""IMGVR_UViG_2904879368_000003_2904879368_2904879379_176713-228057""","""Viral genomes""","""virus_genomes_gut""","""Human gut""",false,true,true
…,…,…,…,…,…,…,…
"""scaffold_9493_c1_8""","""scaffold_9493_c1""","""Viromes""","""soil_T42_15_2_44""","""Soil""",false,false,true
"""scaffold_22_c1_40""","""scaffold_22_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,true,true
"""scaffold_473_c1_10""","""scaffold_473_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,false,true


In [85]:
vibrant_amg_genes.write_parquet(AMG_TABLES_DIR.joinpath("amg_predictions_vibrant.parquet"))

## Combine the AMG tables from each tool into one
One row per gene, combine the columns.

### Get super set of genes
Since not all tools include every gene present in the output of other tools.

In [86]:
amg_genes_super = (
    pl.concat(
        [
            checkamg_amg_genes.select(["gene", "scaffold", "source", "sample", "ecosystem"]),
            dramv_amg_genes.select(["gene", "scaffold", "source", "sample", "ecosystem"]),
            vibrant_amg_genes.select(["gene", "scaffold", "source", "sample", "ecosystem"])
        ],
        how="diagonal_relaxed"
    )
    .unique()
    .sort(["gene", "scaffold", "source", "sample", "ecosystem"])
)

In [87]:
amg_genes_super

gene,scaffold,source,sample,ecosystem
str,str,str,str,str
"""Ga0485157_0000001_10""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic"""
"""Ga0485157_0000001_100""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic"""
"""Ga0485157_0000001_101""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic"""
"""Ga0485157_0000001_102""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic"""
"""Ga0485157_0000001_103""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic"""
…,…,…,…,…
"""scaffold_9_c1_98""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil"""
"""scaffold_9_c1_98""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil"""
"""scaffold_9_c1_99""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_1_55""","""Soil"""


### Join geNomad results to AMG predictions

#### Load geNomad results

In [88]:
GENOMAD_DIR = GENOME_DIR.joinpath("genomad_outputs")
GENOMAD_DIR

PosixPath('/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes/genomad_outputs')

In [89]:
base_dir_genomad = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/benchmark_genomes")
pattern_genomad = "genomad_outputs/*/*/*_summary/*_virus_summary.tsv"
paths_genomad = sorted(base_dir_genomad.glob(pattern_genomad))

In [90]:
dfs_genomad = []
for p in paths_genomad:
    parts = p.parts
    try:
        i = parts.index("genomad_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
        # print(f"Processing geNomad virus file: {p} (source={source}, sample={sample}, ecosystem={ecosystem})")
    except (ValueError, IndexError):
        continue

    try:
        df = pl.read_csv(p, separator="\t", infer_schema_length=200, has_header=True)
    except Exception:
        df = pl.read_csv(p, separator=",", infer_schema_length=200, has_header=True)

    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
        pl.col("seq_name")
          .cast(pl.Utf8)
          .str.replace(r"\|provirus_\d+_\d+$", "", literal=False)
          .alias("seq_name_clean")
    ])

    if "coordinates" in df.columns:
        df = df.with_columns([
            pl.col("coordinates")
              .cast(pl.Utf8)
              .str.extract(r"(\d+)-(\d+)", 1)
              .cast(pl.Int64)
              .alias("provirus_start"),
            pl.col("coordinates")
              .cast(pl.Utf8)
              .str.extract(r"(\d+)-(\d+)", 2)
              .cast(pl.Int64)
              .alias("provirus_end")
        ]).drop("coordinates")
        
    df = (
        df
        .drop("seq_name")
        .rename({"seq_name_clean": "seq_name"})
        .select(
            [
                "source", "sample", "ecosystem", "seq_name",
                "length", "topology",
                "provirus_start", "provirus_end",
                "n_genes", "genetic_code",
                "virus_score", "fdr", "n_hallmarks",
                "marker_enrichment", "taxonomy"
            ]
        )
    )

    dfs_genomad.append(df)

genomad_virus_all = (
    pl.concat(dfs_genomad, how="diagonal_relaxed")
    .unique()
    .sort(
        ["source", "sample", "ecosystem", "fdr"],
        descending=[False, False, False, False]
        )
)
genomad_virus_all = reformat_source_ecosystem_tool(genomad_virus_all)
genomad_virus_all

source,sample,ecosystem,seq_name,length,topology,provirus_start,provirus_end,n_genes,genetic_code,virus_score,fdr,n_hallmarks,marker_enrichment,taxonomy
str,str,str,str,i64,str,i64,i64,i64,i64,f64,f64,i64,f64,str
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_3300045988_025220|3300045988|Ga0495776_012388""",98499,"""DTR""",null,null,82,15,0.9999,0.0001,7,79.086,"""Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_3300007361_000006|3300007361|Ga0104787_100140""",85456,"""No terminal repeats""",null,null,110,11,0.9999,0.0001,11,150.4548,"""Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_3300045988_023097|3300045988|Ga0495776_165444""",47357,"""No terminal repeats""",null,null,70,11,0.9998,0.0001,15,97.625,"""Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_3300045988_054009|3300045988|Ga0495776_073326""",60686,"""No terminal repeats""",null,null,95,11,0.9998,0.0001,11,78.9456,"""Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_3300045988_022999|3300045988|Ga0495776_145193""",33411,"""No terminal repeats""",null,null,49,11,0.9998,0.0001,11,65.891,"""Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_72352_c1""",1011,"""No terminal repeats""",null,null,1,11,0.3763,0.3953,0,0.0,"""Unclassified"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_17845_c1""",1906,"""No terminal repeats""",null,null,3,11,0.3648,0.3953,0,0.0,"""Unclassified"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_94626_c1""",892,"""No terminal repeats""",null,null,1,11,0.3644,0.3953,0,0.0,"""Unclassified"""


#### Join geNomad results to AMG gene superset
Enforcing a constant maximum FDR (from genomad virus prediction) to call a sequence a virus, and a minimum input sequence length, since each tool uses different defaults (CheckAMG was 5000, DRAM-V was 2500, and VIBRANT was 100).

In [91]:
MAX_GENOMAD_FDR = 0.1 # Set a maximum FDR threshold (from geNomad's score-calibration) for virus inclusion

In [92]:
MIN_INPUT_SEQ_LEN = 5000 # Minimum input sequence length to consider

In [93]:
g = genomad_virus_all.select([
    pl.col("source").cast(pl.Utf8),
    pl.col("sample").cast(pl.Utf8),
    pl.col("seq_name").cast(pl.Utf8).alias("genomad_scaffold"),
    pl.col("topology").cast(pl.Utf8),
    pl.col("provirus_start").cast(pl.Int64),
    pl.col("provirus_end").cast(pl.Int64),
    pl.col("fdr").cast(pl.Float64),
])

has_prov = (
    g.filter(pl.col("topology") == "Provirus")
     .select(["source","sample","genomad_scaffold"])
     .unique()
     .with_columns(pl.lit(True).alias("has_provirus"))
)

scaf_fdr = (
    g.filter(pl.col("topology") != "Provirus")
     .group_by(["source","sample","genomad_scaffold"])
     .agg(pl.col("fdr").min().alias("genomad_scaffold_fdr_min"))
)

prov = g.filter(pl.col("topology") == "Provirus").select(
    ["source","sample","genomad_scaffold","provirus_start","provirus_end","fdr"]
)

def attach_genomad(df: pl.DataFrame, scaffold_expr: pl.Expr, start_col: str, end_col: str) -> pl.DataFrame:
    tmp_scaf = "__scaffold_tmp"

    # normalize all pipe characters to underscores in scaffold expression
    norm_scaf_expr = scaffold_expr.cast(pl.Utf8).str.replace_all(r"\|", "_")

    df2 = (
        df.with_columns([
            norm_scaf_expr.alias(tmp_scaf),
            pl.col(start_col).cast(pl.Int64).alias("__start_tmp"),
            pl.col(end_col).cast(pl.Int64).alias("__end_tmp"),
        ])
        # also normalize any existing pipe chars in 'source' and 'sample' just in case
        .with_columns([
            pl.col("source").cast(pl.Utf8).str.replace_all(r"\|", "_"),
            pl.col("sample").cast(pl.Utf8).str.replace_all(r"\|", "_")
        ])
        .with_row_index("__rid")
    )

    # normalize genomad keys the same way
    prov_norm = prov.with_columns(
        pl.col("genomad_scaffold").cast(pl.Utf8).str.replace_all(r"\|", "_").alias("genomad_scaffold")
    )
    scaf_fdr_norm = scaf_fdr.with_columns(
        pl.col("genomad_scaffold").cast(pl.Utf8).str.replace_all(r"\|", "_").alias("genomad_scaffold")
    )
    has_prov_norm = has_prov.with_columns(
        pl.col("genomad_scaffold").cast(pl.Utf8).str.replace_all(r"\|", "_").alias("genomad_scaffold")
    )

    # overlaps with ANY provirus on that scaffold
    overlap = (
        df2.select(["__rid","source","sample",tmp_scaf,"__start_tmp","__end_tmp"])
           .join(
               prov_norm,
               left_on=["source","sample",tmp_scaf],
               right_on=["source","sample","genomad_scaffold"],
               how="left",
           )
           .with_columns([
               (
                   pl.col("__start_tmp").is_not_null() & pl.col("__end_tmp").is_not_null() &
                   pl.col("provirus_start").is_not_null() & pl.col("provirus_end").is_not_null() &
                   (pl.col("__start_tmp") <= pl.col("provirus_end")) &
                   (pl.col("__end_tmp") >= pl.col("provirus_start"))
               ).alias("__overlaps")
           ])
           .filter(pl.col("__overlaps"))
           .group_by("__rid")
           .agg(pl.col("fdr").min().alias("__genomad_provirus_fdr_min"))
    )

    out = (
        df2
        .join(has_prov_norm, left_on=["source","sample",tmp_scaf],
              right_on=["source","sample","genomad_scaffold"], how="left")
        .with_columns(pl.col("has_provirus").fill_null(False))
        .join(overlap, on="__rid", how="left")
        .join(scaf_fdr_norm, left_on=["source","sample",tmp_scaf],
              right_on=["source","sample","genomad_scaffold"], how="left")
        .with_columns([
            pl.when(pl.col("has_provirus"))
            .then(pl.col("__genomad_provirus_fdr_min"))
            .otherwise(pl.col("genomad_scaffold_fdr_min"))
            .alias("genomad_region_fdr"),
        ])
        .with_columns([
            pl.when(pl.col("has_provirus") & pl.col("__genomad_provirus_fdr_min").is_null())
            .then(pl.lit(False))
            .otherwise(
                pl.col("genomad_region_fdr").is_not_null() &
                (pl.col("genomad_region_fdr") < MAX_GENOMAD_FDR)
            )
            .alias("genomad_viral")
        ])
        .with_columns(
            pl.when(pl.col("scaffold_len_bases") < MIN_INPUT_SEQ_LEN)
            .then(pl.lit(False))
            .otherwise(pl.lit(True))
            .alias("passes_min_input_seq_len")
        )
    )

    to_drop = [c for c in [
        "__rid","__start_tmp","__end_tmp","__genomad_provirus_fdr_min",
        "genomad_scaffold_fdr_min", tmp_scaf
    ] if c in out.columns]
    return out.drop(to_drop)

In [94]:
amg_genes_genomad = (
    attach_genomad(
    # join to gene_id_mapping to get gene coordinates and scaffold lengths for genomad annotation
    (
        amg_genes_super
        .join(
            gene_id_mapping.select(["sample", "new_gene", "new_scaffold", "start", "end", "scaffold_len_bases"]),
            left_on=["gene", "scaffold", "sample"],
            right_on=["new_gene", "new_scaffold", "sample"],
            how="left"
        )
    ),
    scaffold_expr=pl.col("scaffold").cast(pl.Utf8),
    start_col="start",
    end_col="end"
    )
    .filter(pl.col("scaffold_len_bases") >= MIN_INPUT_SEQ_LEN)
    .drop(["start", "end", "scaffold_len_bases", "has_provirus", "genomad_region_fdr", "passes_min_input_seq_len"])
)

In [95]:
amg_genes_genomad

gene,scaffold,source,sample,ecosystem,genomad_viral
str,str,str,str,str,bool
"""Ga0485157_0000001_10""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false
"""Ga0485157_0000001_100""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false
"""Ga0485157_0000001_101""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false
"""Ga0485157_0000001_102""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false
"""Ga0485157_0000001_103""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false
…,…,…,…,…,…
"""scaffold_9_c1_98""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""",true
"""scaffold_9_c1_98""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""",true
"""scaffold_9_c1_99""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_1_55""","""Soil""",true


### Join genome context information to AMG genes
Using CheckAMG annotate's genes_genomic_context table, which contains all input genes.

#### Load genome context tables

In [96]:
pattern_context = "*/*/results/genes_genomic_context.parquet"
paths_context = sorted(CHECKAMG_DIR.glob(pattern_context))

dfs_context = []
for p in paths_context:
    parts = p.parts
    try:
        i = parts.index("checkamg_annotate_v1.1_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
    except (ValueError, IndexError):
        continue

    df = pl.read_parquet(p)

    if df is not None:
        df = df.with_columns([
            pl.lit(sample).alias("sample"),
            pl.lit(source).alias("source"),
            pl.lit(ecosystem).alias("ecosystem")
        ])
        
        df = df.join(
            reformat_for_merge(gene_id_mapping),
            left_on=["protein", "contig", "sample"],
            right_on=["gene", "scaffold", "sample"],
            how="left"
            ).rename({"protein": "gene", "contig": "scaffold"})
        df = df.drop("genome")
        df = df.select(
            [
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            ] + [col for col in df.columns if col not in {
                "source", "sample", "ecosystem",
                "gene", "gene_number", "start", "end", "frame",
                "scaffold", "scaffold_n_genes", "scaffold_len_bases",
            }]
        ).sort(
            ["source", "sample", "ecosystem", "scaffold", "start"],
            descending=[False, False, False, False, False]
        )
        
        dfs_context.append(df)

genome_context_raw = pl.concat(dfs_context, how="diagonal_relaxed")
genome_context_raw = reformat_source_ecosystem_tool(genome_context_raw)

In [97]:
genome_context_raw

source,sample,ecosystem,gene,gene_number,start,end,frame,scaffold,scaffold_n_genes,scaffold_len_bases,contig_pos_start,contig_pos_end,is_bin,FOAM_score,PHROG_score,METABOLIC_score,dbCAN_score,Pfam_score,KEGG_score,FOAM_alignment_type,PHROG_alignment_type,METABOLIC_alignment_type,dbCAN_alignment_type,Pfam_alignment_type,KEGG_alignment_type,FOAM_hmm_id,PHROG_hmm_id,METABOLIC_hmm_id,dbCAN_hmm_id,Pfam_hmm_id,KEGG_hmm_id,FOAM_evalue,PHROG_evalue,METABOLIC_evalue,dbCAN_evalue,Pfam_evalue,…,viral_region_id,step1_seed_strict_wvl,step1_in_candidate_region,step2_passes_vscore_gate,step2_in_refined_core,step3_bridge_ok,step3_in_walkback_extended,step4_in_snapped_region,step5_in_merged_region,KEGG_viral_left_dist,KEGG_viral_right_dist,Pfam_viral_left_dist,Pfam_viral_right_dist,PHROG_viral_left_dist,PHROG_viral_right_dist,KEGG_MGE_left_dist,KEGG_MGE_right_dist,Pfam_MGE_left_dist,Pfam_MGE_right_dist,PHROG_MGE_left_dist,PHROG_MGE_right_dist,KEGG_V-score_left_MGE,KEGG_V-score_right_MGE,KEGG_VL-score_left_MGE,KEGG_VL-score_right_MGE,Pfam_V-score_left_MGE,Pfam_V-score_right_MGE,Pfam_VL-score_left_MGE,Pfam_VL-score_right_MGE,PHROG_V-score_left_MGE,PHROG_V-score_right_MGE,PHROG_VL-score_left_MGE,PHROG_VL-score_right_MGE,LGBM_viral_prob,Viral_Origin_Confidence,gene_number_right,frame_right
str,str,str,str,i64,i64,i64,i64,str,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,…,i32,bool,bool,bool,bool,bool,bool,bool,bool,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str,i64,i64
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_1""",1,1,966,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,1,966,"""false""",null,291.70929,null,null,null,150.210114,null,"""full""",null,null,null,"""full""",null,"""phrog_157""",null,null,null,"""K07497""",null,5.6025e-86,null,null,null,…,0,true,true,true,true,true,true,true,true,null,966.0,null,966.0,null,2658.0,null,2658.0,null,null,null,2658.0,null,10.0,null,4.373298,null,null,null,null,null,10.0,null,4.311987,0.28029,"""low""",1,-1
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_2""",2,963,1640,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,963,1640,"""false""",null,null,null,null,100.774567,null,null,null,null,null,"""full""",null,null,null,null,null,"""PF13518.13""",null,null,null,null,null,1.1915e-28,…,0,true,true,false,true,true,true,true,true,966.0,1692.0,966.0,1692.0,966.0,1692.0,966.0,1692.0,null,null,966.0,1692.0,10.0,10.0,3.75473,4.373298,null,null,null,null,10.0,10.0,3.078457,4.311987,0.344168,"""low""",2,-1
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_3""",3,1796,2809,1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,1796,2809,"""false""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,0,true,true,false,true,true,true,true,true,678.0,1014.0,678.0,1014.0,1644.0,1014.0,1644.0,1014.0,null,null,1644.0,1014.0,10.0,10.0,3.75473,4.373298,null,null,null,null,10.0,10.0,3.078457,4.311987,0.376621,"""low""",3,1
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_4""",4,2657,4198,-1,"""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525""",57,35665,2657,4198,"""false""",null,333.477875,null,null,null,302.550446,null,"""full""",null,null,null,"""full""",null,"""phrog_95""",null,null,null,"""K06400""",null,1.0745e-98,null,null,null,…,0,true,true,true,true,true,true,true,true,1692.0,2544.0,1692.0,2544.0,2658.0,2544.0,2658.0,null,null,null,2658.0,null,10.0,null,3.75473,null,null,null,null,null,10.0,null,3.078457,null,0.912352,"""hig

In [98]:
genome_context = (
    genome_context_raw
    .with_columns([
        # Create a column "viral_gene_left_dist" and "viral_gene_right_dist" that is the minimum value
        # for all columns ending with "_viral_left_dist" or "_viral_right_dist"
        pl.min_horizontal(*[col for col in genome_context_raw.columns if col.endswith("_viral_left_dist")]).alias("viral_gene_left_dist"),
        pl.min_horizontal(*[col for col in genome_context_raw.columns if col.endswith("_viral_right_dist")]).alias("viral_gene_right_dist"),
        # Create a column "MGE_gene_left_dist" and "MGE_gene_right_dist" that is the minimum value
        # for all columns ending with "_MGE_left_dist" or "_MGE_right_dist"
        pl.min_horizontal(*[col for col in genome_context_raw.columns if col.endswith("_MGE_left_dist")]).alias("MGE_gene_left_dist"),
        pl.min_horizontal(*[col for col in genome_context_raw.columns if col.endswith("_MGE_right_dist")]).alias("MGE_gene_right_dist"),
        # Create a column "MGE_gene_left_V_score" and "MGE_gene_right_V_score" that is the maximum value
        # for all columns ending with "_V-score_left_MGE" or "_V-score_right_MGE"
        pl.max_horizontal(*[col for col in genome_context_raw.columns if col.endswith("_V-score_left_MGE")]).alias("MGE_gene_left_V_score"),
        pl.max_horizontal(*[col for col in genome_context_raw.columns if col.endswith("_V-score_right_MGE")]).alias("MGE_gene_right_V_score"),
    ])
    .select(
        [
            "source", "sample", "ecosystem", "gene", "scaffold",
            "viral_gene_left_dist", "viral_gene_right_dist",
            "MGE_gene_left_dist", "MGE_gene_right_dist",
            "MGE_gene_left_V_score", "MGE_gene_right_V_score",
            "step5_in_merged_region",
            "start", "end"
            ]
    )
    .rename({"step5_in_merged_region": "in_strict_viral_region"})
    .unique() # deduplicate in case of redundant joins
    .sort(
        ["source", "sample", "ecosystem", "scaffold", "start", "end"],
        descending=[False, False, False, False, False, False]
    )
    .drop(["start", "end"])
)

In [99]:
genome_context

source,sample,ecosystem,gene,scaffold,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,in_strict_viral_region
str,str,str,str,str,f32,f32,f32,f32,f32,f32,bool
"""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""Ga0485157_0000001_1""","""Ga0485157_0000001""",null,3780.0,null,null,null,null,false
"""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""Ga0485157_0000001_2""","""Ga0485157_0000001""",null,3192.0,null,null,null,null,false
"""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""Ga0485157_0000001_3""","""Ga0485157_0000001""",null,1662.0,null,null,null,null,false
"""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""Ga0485157_0000001_4""","""Ga0485157_0000001""",null,5259.0,null,null,null,null,false
"""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""Ga0485157_0000001_5""","""Ga0485157_0000001""",1824.0,3435.0,null,null,null,null,false
…,…,…,…,…,…,…,…,…,…,…,…
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_30""","""scaffold_9_c1""",1101.0,1386.0,null,1386.0,null,10.0,true
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_31""","""scaffold_9_c1""",1017.0,369.0,null,369.0,null,10.0,true
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_32""","""scaffold_9_c1""",1386.0,1422.0,null,null,null,null,true


In [100]:
amg_genes_genome_context = (
    amg_genes_genomad
    .join(
        genome_context,
        on=["source", "sample", "ecosystem", "gene", "scaffold"],
        how="left"
    )
)

In [101]:
amg_genes_genome_context

gene,scaffold,source,sample,ecosystem,genomad_viral,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,in_strict_viral_region
str,str,str,str,str,bool,f32,f32,f32,f32,f32,f32,bool
"""Ga0485157_0000001_10""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,5259.0,3531.0,null,null,null,null,false
"""Ga0485157_0000001_100""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,12252.0,6792.0,null,null,null,null,false
"""Ga0485157_0000001_101""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,594.0,6198.0,null,null,null,null,false
"""Ga0485157_0000001_102""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,816.0,5976.0,null,null,null,null,false
"""Ga0485157_0000001_103""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,1161.0,5631.0,null,null,null,null,false
…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_98""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""",true,4608.0,1272.0,null,null,null,null,true
"""scaffold_9_c1_98""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""",true,2829.0,7764.0,null,29667.0,null,10.0,true
"""scaffold_9_c1_99""","""scaffold_9_c1""","""Mixed metagenomes""","""soil_T42_15_1_55""","""Soil""",true,19902.0,1983.0,21198.0,null,10.0,null,false


In [102]:
amg_genes = (
    amg_genes_genome_context
    .join(checkamg_amg_genes, on=["gene", "scaffold", "source", "sample", "ecosystem"], how="left")
    .join(dramv_amg_genes, on=["gene", "scaffold", "source", "sample", "ecosystem"], how="left")
    .join(vibrant_amg_genes, on=["gene", "scaffold", "source", "sample", "ecosystem"], how="left")
    .unique()
    .fill_null(False)
    .sort(["source", "sample", "ecosystem", "gene"])
)

In [103]:
amg_genes

gene,scaffold,source,sample,ecosystem,genomad_viral,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,in_strict_viral_region,CheckAMG_high,CheckAMG_medium,CheckAMG_low,DRAMV_default,DRAMV_allow_T,DRAMV_aux_4,DRAMV_aux_4_allow_T,DRAMV_auxiliary_score,DRAMV_M,DRAMV_T,DRAMV_V,DRAMV_A,DRAMV_P,DRAMV_B,DRAMV_F,VIBRANT_high,VIBRANT_medium,VIBRANT_low
str,str,str,str,str,bool,f32,f32,f32,f32,f32,f32,bool,bool,bool,bool,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""Ga0485157_0000001_10""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,5259.0,3531.0,null,null,null,null,false,false,false,true,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_100""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,12252.0,6792.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_101""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,594.0,6198.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_102""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,816.0,5976.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_103""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,1161.0,5631.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_5""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,5847.0,4218.0,null,47403.0,null,10.0,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""scaffold_9_c1_6""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,6696.0,3369.0,null,46554.0,null,10.0,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""scaffold_9_c1_7""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,7674.0,2391.0,null,45576.0,null,10.0,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false


In [104]:
amg_genes.write_parquet(AMG_TABLES_DIR.joinpath("amg_predictions_combined.parquet"))